# JAX evaluation for Granite reasoning tests

This notebook adds a **JAX-based analysis layer** to `granite-finreason-lab`.

Granite inference still runs through the repo's existing PyTorch / Transformers scripts. JAX is used **after inference** for fast numerical evaluation: accuracy, latency, token efficiency, vectorized statistics, and bootstrap confidence intervals.

The notebook expects the existing JSONL schema from `results/exp_*.jsonl`. If no result file is present, it uses a tiny built-in demo dataset so every cell still runs.


## 1. Imports

Colab normally includes JAX. The cell below installs it only when it is missing.


In [ ]:
import importlib.util
import json
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("jax") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "jax[cpu]>=0.4.30"])

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import pandas as pd

print("JAX version:", jax.__version__)
print("Devices:", jax.devices())


## 2. Load a Granite experiment result

Run one of the repo experiments first if you want real data, for example:

```bash
python experiments/01_sequential_arithmetic/run.py \
  --model ibm-granite/granite-4.2-3b-instruct \
  --depths 2 4 8 12 16 24 32 \
  --samples-per-depth 100
```


In [ ]:
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

result_files = sorted((repo_root / "results").glob("exp_*.jsonl"))

if result_files:
    result_path = result_files[-1]
    with result_path.open() as f:
        records = [json.loads(line) for line in f if line.strip()]
    print(f"Loaded {len(records)} records from {result_path}")
else:
    print("No results/exp_*.jsonl found — using a tiny demo dataset.")
    records = [
        {"task_id": "demo_01", "depth": 2, "thinking_mode": "non_thinking", "correct": True,  "generated_tokens": 18,  "latency_s": 0.31},
        {"task_id": "demo_02", "depth": 2, "thinking_mode": "non_thinking", "correct": True,  "generated_tokens": 19,  "latency_s": 0.33},
        {"task_id": "demo_03", "depth": 8, "thinking_mode": "non_thinking", "correct": False, "generated_tokens": 24,  "latency_s": 0.40},
        {"task_id": "demo_04", "depth": 8, "thinking_mode": "non_thinking", "correct": True,  "generated_tokens": 25,  "latency_s": 0.41},
        {"task_id": "demo_05", "depth": 2, "thinking_mode": "full_thinking", "correct": True,  "generated_tokens": 94,  "latency_s": 0.88},
        {"task_id": "demo_06", "depth": 2, "thinking_mode": "full_thinking", "correct": True,  "generated_tokens": 87,  "latency_s": 0.82},
        {"task_id": "demo_07", "depth": 8, "thinking_mode": "full_thinking", "correct": True,  "generated_tokens": 161, "latency_s": 1.37},
        {"task_id": "demo_08", "depth": 8, "thinking_mode": "full_thinking", "correct": True,  "generated_tokens": 149, "latency_s": 1.29},
    ]

df = pd.DataFrame(records)
df[["task_id", "depth", "thinking_mode", "correct", "generated_tokens", "latency_s"]].head()


## 3. Core metrics in JAX

`jax.jit` compiles the numerical part. This is deliberately separated from pandas/file I/O.


In [ ]:
@jax.jit
def accuracy(correct: jax.Array) -> jax.Array:
    return jnp.mean(correct)


@jax.jit
def mean_latency(latency_s: jax.Array) -> jax.Array:
    return jnp.mean(latency_s)


@jax.jit
def mean_tokens(generated_tokens: jax.Array) -> jax.Array:
    return jnp.mean(generated_tokens)


@jax.jit
def tokens_per_correct_answer(correct: jax.Array, generated_tokens: jax.Array) -> jax.Array:
    correct_count = jnp.maximum(jnp.sum(correct), 1.0)
    return jnp.sum(generated_tokens) / correct_count


### Small executable tests

These assertions make sure the JAX metric functions behave as expected before we apply them to model results.


In [ ]:
toy_correct = jnp.array([1.0, 0.0, 1.0, 1.0])
toy_latency = jnp.array([1.0, 2.0, 3.0, 4.0])
toy_tokens = jnp.array([10.0, 20.0, 30.0, 40.0])

assert jnp.isclose(accuracy(toy_correct), 0.75)
assert jnp.isclose(mean_latency(toy_latency), 2.5)
assert jnp.isclose(mean_tokens(toy_tokens), 25.0)
assert jnp.isclose(tokens_per_correct_answer(toy_correct, toy_tokens), 100.0 / 3.0)

print("All JAX metric tests passed.")


## 4. Bootstrap uncertainty with JAX

A raw accuracy such as 78% is much more useful when accompanied by uncertainty. Here JAX performs thousands of bootstrap resamples in one vectorized operation.


In [ ]:
def bootstrap_accuracy_ci(key: jax.Array, correct: jax.Array, n_boot: int = 5000):
    n = correct.shape[0]
    indices = jax.random.randint(key, shape=(n_boot, n), minval=0, maxval=n)
    boot_accuracy = jnp.mean(correct[indices], axis=1)
    return jnp.quantile(boot_accuracy, jnp.array([0.025, 0.975]))


bootstrap_accuracy_ci_jit = jax.jit(bootstrap_accuracy_ci, static_argnames=("n_boot",))

demo_ci = bootstrap_accuracy_ci_jit(jax.random.PRNGKey(42), toy_correct)
print("Toy 95% CI:", [float(x) for x in demo_ci])


## 5. Evaluate each thinking mode at each reasoning depth

This directly matches the central question of the repo: **how does accuracy change as required reasoning depth increases, and what do extra reasoning tokens buy us?**


In [ ]:
rows = []
key = jax.random.PRNGKey(2026)

for (mode, depth), group in df.groupby(["thinking_mode", "depth"], sort=True):
    correct = jnp.asarray(group["correct"].astype(float).to_numpy())
    latency = jnp.asarray(group["latency_s"].astype(float).to_numpy())
    tokens = jnp.asarray(group["generated_tokens"].astype(float).to_numpy())

    key, subkey = jax.random.split(key)
    ci_low, ci_high = bootstrap_accuracy_ci_jit(subkey, correct)

    rows.append({
        "thinking_mode": mode,
        "depth": int(depth),
        "n": len(group),
        "accuracy": float(accuracy(correct)),
        "accuracy_ci_low": float(ci_low),
        "accuracy_ci_high": float(ci_high),
        "mean_latency_s": float(mean_latency(latency)),
        "mean_generated_tokens": float(mean_tokens(tokens)),
        "tokens_per_correct_answer": float(tokens_per_correct_answer(correct, tokens)),
    })

summary = pd.DataFrame(rows).sort_values(["thinking_mode", "depth"]).reset_index(drop=True)
summary


## 6. Accuracy-vs-depth curve

With real experiment output, this is the first plot to inspect for the predicted nonlinear failure boundary.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for mode, group in summary.groupby("thinking_mode"):
    group = group.sort_values("depth")
    ax.plot(group["depth"], group["accuracy"], marker="o", label=mode)

ax.set_xlabel("Reasoning depth")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Granite accuracy vs reasoning depth")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## 7. What JAX is buying us here

For a single 100-sample run, ordinary NumPy would already be fast enough. The point of JAX here is to make the evaluation layer ready for heavier numerical work:

- `jit`: compile repeated metrics and resampling workloads.
- vectorized bootstrap: thousands of statistical resamples without Python loops.
- GPU/TPU execution when arrays and evaluation suites become large.
- a natural next step toward calibration metrics, NLL/perplexity, pairwise model comparisons, and larger model × mode × depth sweeps.

A useful next extension is to store token-level log probabilities during inference. Then this notebook can add **NLL, perplexity, Brier score and calibration** rather than evaluating only correctness, latency and generated-token cost.
